# Novel-level keyness (v2)

For each word in the environmental lexicon, finds which novels use that word unusually
often relative to the rest of the corpus (Dunning G2 log-likelihood, novel-vs-corpus --
same test as `tables/freq_shift_g2.csv` in `w2v_export_v2.py`, just applied per-book
instead of per-era).

**Notebook, not a script, on purpose.** The `scripts/novel_keyness_v2.py` version ran to
completion (printed correct row counts) but the actual output files never landed on
disk -- and by the time that was discovered, the ~4-hour corpus-cleaning pass had to be
redone from scratch to retry a small file write. This notebook separates the SLOW step
(cleaning + loading every novel) from the FAST step (scoring, writing, and *verifying*
the write actually landed) into different cells, so a write failure only costs a re-run
of the cheap cells, not the whole corpus pass. Every write cell is followed by a
verify cell that re-reads the file fresh from disk and checks it against what was
supposed to be there -- a `print("... rows")` right after `.to_csv()` only proves the
in-memory DataFrame was correct, not that the write actually persisted.

Must run **inside the capsule, in secure mode** -- raw per-novel text only exists at
`TEXT_DIR` on the secure volume and never leaves it.

## Imports

In [ ]:
import json
import math
import os
import re
from collections import Counter
from multiprocessing import get_context
from pathlib import Path

import pandas as pd

## OCR cleaning

Copied verbatim from `SF_word2vec_eras_v2.ipynb` (cell `46a9ca82`) / `scripts/novel_keyness_v2.py`,
so word counts here are directly comparable to the word2vec pipeline's.

In [ ]:
_norm_ws = re.compile(r"\s+")
_only_number = re.compile(r"^\s*[\divxlcIVXLC]+\s*$")
_hyphen_break = re.compile(r"(\w)-\s*\n\s*(\w)")
_word = re.compile(r"[A-Za-z']+")


def discover_volumes(base_dir, ids=None):
    vols = {}
    for d in sorted(p for p in Path(base_dir).iterdir() if p.is_dir() and not p.name.startswith(".")):
        if ids is not None and d.name not in ids:
            continue
        pages = sorted(d.glob("*.txt"))
        if pages:
            vols[d.name] = pages
    return vols


def _norm_line(line):
    return _norm_ws.sub(" ", re.sub(r"\d+", "", line)).strip().lower()


def _volume_vocab(page_paths):
    words = set()
    for p in page_paths:
        text, _ = _hyphen_break.subn(r"\1\2", p.read_text(encoding="utf-8", errors="replace"))
        words.update(w.lower() for w in _word.findall(text) if len(w) > 1)
    return words


def build_dictionary(volumes, min_vols, procs=None):
    df = Counter()
    with get_context("fork").Pool(procs or min(16, os.cpu_count() or 1)) as pool:
        for vocab in pool.imap_unordered(_volume_vocab, list(volumes.values()), chunksize=8):
            df.update(vocab)
    return {w for w, c in df.items() if c >= min_vols}


def page_dict_rate(text, dictionary):
    words = [w.lower() for w in _word.findall(text) if len(w) > 1]
    return sum(1 for w in words if w in dictionary) / len(words) if words else 0.0


def clean_volume(page_paths, dictionary, running_head_min_pages, running_head_frac,
                  running_head_max_chars, page_min_dict_rate):
    pages = [p.read_text(encoding="utf-8", errors="replace") for p in page_paths]
    line_pages = Counter()
    per_page_lines = []
    for text in pages:
        lines = text.split("\n")
        per_page_lines.append(lines)
        line_pages.update({_norm_line(l) for l in lines if 0 < len(l.strip()) <= running_head_max_chars})
    thresh = max(running_head_min_pages, int(running_head_frac * len(pages)))
    heads = {l for l, c in line_pages.items() if c >= thresh and l}
    kept = []
    for lines in per_page_lines:
        out = []
        for l in lines:
            if (len(l.strip()) <= running_head_max_chars and _norm_line(l) in heads) or _only_number.match(l):
                continue
            out.append(l)
        page_text, _ = _hyphen_break.subn(r"\1\2", "\n".join(out))
        if dictionary is not None and page_min_dict_rate and page_dict_rate(page_text, dictionary) < page_min_dict_rate:
            continue
        kept.append(page_text)
    text, _ = _hyphen_break.subn(r"\1\2", "\n".join(kept))
    return text


def load_docs_keep_id(base_dir, dictionary, ids=None, **clean_kwargs):
    """Like the word2vec notebook's load_docs(), but keeps {htid: cleaned_text} instead of discarding htid."""
    docs = {}
    for htid, pages in discover_volumes(base_dir, ids).items():
        docs[htid] = clean_volume(pages, dictionary, **clean_kwargs)
    return docs

## Era assignment

Copied from `SF_word2vec_eras_v2.ipynb` (cell `7030de76`).

In [ ]:
def assign_era(year, cutoffs=(1962, 1972), labels=("era_a", "era_b", "era_c")):
    for cutoff, label in zip(cutoffs, labels):
        if year < cutoff:
            return label
    return labels[-1]

## Metadata loading

A handful of `htid`s in `metadata_august2026.csv` are genuine Ace Doubles / omnibus scans --
one physical volume, one `htid`, two or three distinct novels (sometimes by different authors)
bound together -- not data-entry duplicates. `year` is identical across every such group
(checked); `title` and `author` are joined with `" / "` rather than picking one row and
silently losing the other(s).

In [ ]:
def load_metadata(path):
    df = pd.read_csv(path)
    df["htid"] = df["htid"].astype(str)

    def join_unique(values):
        return " / ".join(dict.fromkeys(str(v) for v in values))

    grouped = df.groupby("htid").agg(
        title=("title", join_unique),
        author=("author", join_unique),
        year=("year", "first"),
    )
    return grouped.to_dict("index")

## Lexicon (v2, reviewed 2026-09-23)

101 words, same list as `scripts/novel_keyness_v2.py` and `scripts/keyword_context_v2.py`
-- `war` added to `human_agency`; `power`/`cycle`/`ice`/`cistern`/`culvert` dropped as too
generic. This is a *separate* lexicon from the word2vec notebook's own `ENV_WORD_GROUPS`
(105 words) -- that one intentionally stayed on the original list so the embeddings
analysis stays consistent with results already produced from it. See
`SF_word2vec_eras_v2.ipynb`'s keyword-list markdown cell for the full review history.

In [ ]:
ENV_WORD_GROUPS = {
    "landscape_baseline": ["river", "creek", "stream", "water", "forest", "nature", "wilderness", "jungle",
                            "ocean", "landscape", "levee", "dam", "reservoir", "estuary", "wetland",
                            "marsh", "watershed"],
    "ecology_concept": ["ecology", "ecosystem", "environment", "biosphere", "habitat", "balance"],
    "contamination": ["contamination", "waste", "smog", "fumes", "chemical", "pesticide", "insecticide",
                       "pollutant", "exhaust", "toxic", "polluted", "pollution"],
    "waste_infrastructure": ["sewer", "sewage", "drainage", "effluent", "runoff", "wastewater",
                              "cesspool", "sludge", "septic", "plumbing"],
    "population_scarcity": ["overpopulation", "population", "famine", "scarcity", "starvation", "resource", "drought"],
    "energy": ["oil", "fuel", "energy", "coal"],
    "nuclear_atomic": ["radiation", "radioactive", "fallout", "nuclear", "atomic", "bomb", "meltdown"],
    "cosmic_natural_causation": ["solar", "cosmic", "celestial", "geological", "planetary"],
    "human_agency": ["mankind", "humanity", "civilization", "industrial", "war"],
    "disaster_collapse": ["wasteland", "extinction", "collapse", "barren", "dying", "decay", "catastrophe",
                           "apocalypse", "plague"],
    "climate_weather": ["climate", "weather", "warming", "greenhouse", "atmosphere", "temperature",
                         "flood", "flooding", "storm", "hurricane", "glacier", "carbon", "ozone"],
    "space_earth_framing": ["earth", "homeworld", "colony", "frontier", "terraform", "alien"],
}
ENV_WORDS = sorted({w for group in ENV_WORD_GROUPS.values() for w in group})
print(f"{len(ENV_WORDS)} words")

## G2 log-likelihood keyness

Same test as `tables/freq_shift_g2.csv`, novel-vs-corpus instead of era-vs-era.

In [ ]:
def log_likelihood_g2(a, b, c, d):
    """a = count of word in doc, b = count in rest of corpus,
    c = total tokens in doc, d = total tokens in rest of corpus."""
    total_word = a + b
    total_tok = c + d
    if total_word == 0 or total_tok == 0:
        return 0.0
    e1 = c * total_word / total_tok
    e2 = d * total_word / total_tok
    g2 = 0.0
    if a > 0 and e1 > 0:
        g2 += a * math.log(a / e1)
    if b > 0 and e2 > 0:
        g2 += b * math.log(b / e2)
    return 2 * g2

## Config

In [ ]:
TEXT_DIR = "/media/secure_volume/fa50b375-3216-4edd-a685-98488562b723"
METADATA_CSV = "/home/dcuser/Desktop/Clifi-htrc/notebooks/metadata_august2026.csv"
OUT_DIR = Path("/media/secure_volume/out_novel_keyness_v2")
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

MIN_COUNT = 3          # minimum occurrences of a word in a novel to be scored
TOPN = 15               # top N novels per word to keep
DICT_MIN_VOLS = 10
PAGE_MIN_DICT_RATE = 0.55
RUNNING_HEAD_MIN_PAGES = 3
RUNNING_HEAD_FRAC = 0.05
RUNNING_HEAD_MAX_CHARS = 60

## Load metadata

In [ ]:
meta_by_id = load_metadata(METADATA_CSV)
print(f"{len(meta_by_id):,} unique htids after grouping Ace Doubles / omnibus scans")

## Discover volumes + build corpus dictionary

In [ ]:
all_volumes = discover_volumes(TEXT_DIR)
print(f"found {len(all_volumes):,} volumes")

dictionary = build_dictionary(all_volumes, DICT_MIN_VOLS)
print(f"dictionary: {len(dictionary):,} words appear in >= {DICT_MIN_VOLS} volumes")

## Clean + load every volume -- THE SLOW STEP

This is the expensive part (OCR cleaning across the whole corpus). Once this cell finishes,
`docs` stays in kernel memory -- everything below this point can be re-run freely without
paying this cost again, including if a later write turns out to have failed.

In [ ]:
docs = load_docs_keep_id(
    TEXT_DIR, dictionary,
    running_head_min_pages=RUNNING_HEAD_MIN_PAGES,
    running_head_frac=RUNNING_HEAD_FRAC,
    running_head_max_chars=RUNNING_HEAD_MAX_CHARS,
    page_min_dict_rate=PAGE_MIN_DICT_RATE,
)
print(f"loaded {len(docs):,} cleaned documents")

## Per-document word counts

In [ ]:
doc_counts = {}
doc_totals = {}
for htid, text in docs.items():
    toks = [w.lower() for w in _word.findall(text) if len(w) > 1]
    doc_counts[htid] = Counter(toks)
    doc_totals[htid] = len(toks)

corpus_total = sum(doc_totals.values())
corpus_counts = Counter()
for c in doc_counts.values():
    corpus_counts.update(c)

print(f"corpus: {corpus_total:,} tokens across {len(docs):,} novels")

## Score every word against every novel

In [ ]:
rows = []
for word in ENV_WORDS:
    word_corpus_count = corpus_counts.get(word, 0)
    if word_corpus_count == 0:
        continue
    for htid, counts in doc_counts.items():
        a = counts.get(word, 0)
        if a < MIN_COUNT:
            continue
        c = doc_totals[htid]
        b = word_corpus_count - a
        d = corpus_total - c
        rate_doc = a / c if c else 0.0
        rate_rest = b / d if d else 0.0
        if rate_doc <= rate_rest:
            continue  # only keep novels where the word is OVER-represented
        g2 = log_likelihood_g2(a, b, c, d)
        info = meta_by_id.get(htid, {})
        rows.append({
            "word": word,
            "htid": htid,
            "title": info.get("title"),
            "author": info.get("author"),
            "year": info.get("year"),
            "era": assign_era(info["year"]) if pd.notna(info.get("year")) else None,
            "count_in_novel": a,
            "novel_total_tokens": c,
            "rate_per_10k_in_novel": round(rate_doc * 10000, 2),
            "rate_per_10k_rest_of_corpus": round(rate_rest * 10000, 2),
            "g2": round(g2, 2),
        })

result = pd.DataFrame(rows)
result = result.sort_values(["word", "g2"], ascending=[True, False])
top = result.groupby("word", group_keys=False).head(TOPN)
print(f"{len(top):,} rows ready to write (top {TOPN} per word, {top['word'].nunique()} of {len(ENV_WORDS)} words with any hits)")

## Write the CSV

Safe to re-run this cell alone (and the verify cell after it) as many times as needed --
`docs`, `result`, and `top` are already sitting in memory from the cells above.

In [ ]:
csv_path = OUT_DIR / "tables" / "novel_keyness_top_by_word.csv"
top.to_csv(csv_path, index=False)
print(f"wrote {csv_path}")

## Verify the CSV actually landed

Re-reads the file fresh from disk (not from the `top` variable already in memory) and
checks it against what was supposed to be there. This is the check that today's script
run skipped -- its "N rows" message only ever reported the in-memory DataFrame, never
confirmed the write was durable.

In [ ]:
reread = pd.read_csv(csv_path)
assert len(reread) == len(top), f"MISMATCH: wrote {len(top)} rows but disk shows {len(reread)}"
assert reread["word"].nunique() == top["word"].nunique()
print(f"VERIFIED: {csv_path} has {len(reread):,} rows on disk, matches expected {len(top):,}.")

## Write + verify the manifest

In [ ]:
manifest = {
    "n_novels": len(docs),
    "corpus_total_tokens": corpus_total,
    "n_seed_words": len(ENV_WORDS),
    "min_count": MIN_COUNT,
    "topn": TOPN,
    "words_with_no_hits": sorted(set(ENV_WORDS) - set(result["word"].unique())),
}
manifest_path = OUT_DIR / "MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"wrote {manifest_path}")

reread_manifest = json.loads(manifest_path.read_text())
assert reread_manifest == manifest, "MISMATCH: manifest on disk does not match what was written"
print(f"VERIFIED: {manifest_path} matches on disk.")
print(f"{len(manifest['words_with_no_hits'])} words had no qualifying novel: {manifest['words_with_no_hits']}")

## Release

Only run this once both verify cells above have printed `VERIFIED` -- aggregate data
only (counts, scores, titles/authors/years), no raw text. `add` and `done` are kept in
separate cells so you can read the `add` output (what got queued, its size) before
committing to `done`, which finalizes the submission.

In [ ]:
import subprocess


def run_releaseresults(*args):
    cmd = ["releaseresults", *args]
    print("$ " + " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"releaseresults exited with code {result.returncode}")
    return result

In [ ]:
run_releaseresults("add", str(OUT_DIR / "tables"), str(OUT_DIR / "MANIFEST.json"))

Check the output above looks right, then run this to finalize:

In [ ]:
run_releaseresults("done")